In [1]:
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
import pandas as pd


In [2]:

# Load the posts data
posts_path = "/home/ale/Documents/uni/mp/data/posting_behaviour/cleaned/chunk_0_posts_cleaned.parquet"
posts_table = pq.read_table(posts_path)

print("Posts data schema:")
print(posts_table.schema)
print(f"Total posts: {len(posts_table)}")
print("\nFirst few rows of posts:")
print(posts_table.to_pandas().head())

# Load the profiles data
profiles_path = "/home/ale/Documents/uni/mp/data/posting_behaviour/cleaned/profiles_cleaned.parquet"
profiles_table = pq.read_table(profiles_path)

print("\nProfiles data schema:")
print(profiles_table.schema)
print(f"Total profiles: {len(profiles_table)}")
print("\nFirst few rows of profiles:")
print(profiles_table.to_pandas().head())

Posts data schema:
did_id: int64
created_at: timestamp[us, tz=UTC]
Total posts: 5000000

First few rows of posts:
     did_id                       created_at
0  31955122 2023-09-16 19:36:58.558000+00:00
1  31955122 2024-11-15 16:22:35.542000+00:00
2  31955122 2024-11-16 21:00:29.703000+00:00
3  31955122 2024-11-22 20:30:39.146000+00:00
4  31955122 2024-12-13 18:34:28.434000+00:00

Profiles data schema:
did_id: int64
created_at: timestamp[us, tz=UTC]
joined_via_starter_pack: extension<arrow.json>
Total profiles: 32170299

First few rows of profiles:
   did_id                       created_at joined_via_starter_pack
0       1 2024-11-15 07:04:16.352000+00:00                    None
1       2 2024-11-27 18:21:32.298000+00:00                    None
2       3 2024-11-16 13:04:51.383000+00:00                    None
3       4 2024-11-27 14:06:44.280000+00:00                    None
4       5 2024-12-08 15:24:03.663000+00:00                    None


Filter users with less than three posts.

In [3]:
posts_df = posts_table.to_pandas()

print(f"Initial posts count: {len(posts_df)}")
print(f"Unique users: {posts_df['did_id'].nunique()}")

# Convert created_at to datetime if needed
if not pd.api.types.is_datetime64_any_dtype(posts_df['created_at']):
    posts_df['created_at'] = pd.to_datetime(posts_df['created_at'])

# Group by user and count posts
user_post_counts = posts_df.groupby('did_id').size().reset_index(name='post_count')

# Filter users with at least 3 posts
active_users = user_post_counts[user_post_counts['post_count'] >= 3]['did_id']
filtered_posts_df = posts_df[posts_df['did_id'].isin(active_users)]

print(f"Posts after filtering users with <3 posts: {len(filtered_posts_df)}")
print(f"Active users (≥3 posts): {len(active_users)}")

# Sort by user and timestamp for time series creation
filtered_posts_df = filtered_posts_df.sort_values(['did_id', 'created_at'])

Initial posts count: 5000000
Unique users: 130113
Posts after filtering users with <3 posts: 4929439
Active users (≥3 posts): 76264


Create a dictionary with the users join date. 

In [4]:
# Get unique users from our filtered posts
unique_users = filtered_posts_df['did_id'].unique()
print(f"Processing {len(unique_users)} users...")

# Load only the necessary profiles (much smaller)
profiles_df = profiles_table.to_pandas()
if not pd.api.types.is_datetime64_any_dtype(profiles_df['created_at']):
    profiles_df['created_at'] = pd.to_datetime(profiles_df['created_at'])

# Filter profiles to only include our active users
active_profiles = profiles_df[
    profiles_df['did_id'].isin(active_users) & 
    profiles_df['created_at'].notna()
][['did_id', 'created_at']].rename(columns={'created_at': 'join_date'})

print(f"Active users with valid join dates: {len(active_profiles)}")

# Convert to dictionary for fast lookup (much more memory efficient)
join_dates_dict = dict(zip(active_profiles['did_id'], active_profiles['join_date']))
del profiles_df

from itertools import islice

N = 10  # number of rows to print
for did, date in islice(join_dates_dict.items(), N):
    print(f"{did}: {date}")

Processing 76264 users...
Active users with valid join dates: 52170
318: 2024-08-30 21:04:26.303000+00:00
1032: 2024-10-19 12:08:26.894000+00:00
1738: 2024-11-24 22:41:30.245000+00:00
3316: 2024-06-08 03:54:37.834000+00:00
3360: 2024-11-17 05:41:26.862000+00:00
5715: 2024-11-18 23:03:02.662000+00:00
6047: 2024-11-06 16:03:17.668000+00:00
6788: 2025-01-11 13:15:27.369000+00:00
7601: 2024-08-31 02:41:50.895000+00:00
7731: 2024-09-01 18:31:39.795000+00:00


In [5]:
# Merge join dates with posts (more efficient for time series creation)
filtered_posts_with_join = filtered_posts_df.merge(
    active_profiles, on='did_id', how='inner'
)

print("\nStep 2: Calculating days since join and creating daily time series...")

# Calculate days since joining
filtered_posts_with_join['days_since_join'] = (
    (filtered_posts_with_join['created_at'] - filtered_posts_with_join['join_date']).dt.total_seconds() / (24 * 3600)
).round().astype(int)

# Filter to first week only (days 0-6)
first_week_posts = filtered_posts_with_join[
    (filtered_posts_with_join['days_since_join'] >= 0) & 
    (filtered_posts_with_join['days_since_join'] <= 14)
]

print(f"Posts in first week after joining: {len(first_week_posts)}")
print(f"Users with posts in first week: {first_week_posts['did_id'].nunique()}")

# Create daily post counts for each user
daily_post_counts = first_week_posts.groupby(['did_id', 'days_since_join']).size().reset_index(name='post_count')

print(f"Daily post count records: {len(daily_post_counts)}")
print("\nSample of daily post counts:")
print(daily_post_counts.head(10))


Step 2: Calculating days since join and creating daily time series...
Posts in first week after joining: 468324
Users with posts in first week: 39856
Daily post count records: 155047

Sample of daily post counts:
   did_id  days_since_join  post_count
0     318                0           2
1     318                1           5
2     318                2           4
3     318                3           4
4     318                4           3
5     318                5           3
6     318                7           3
7    1738                0          19
8    3316                0           5
9    3316                3           1


In [7]:
print("\nStep 3: Creating wide-format time series dataset...")

# Pivot to create wide format (one row per user, columns for each day)
time_series_wide = daily_post_counts.pivot_table(
    index='did_id', 
    columns='days_since_join', 
    values='post_count', 
    fill_value=0
).reset_index()

# Rename columns to be more descriptive
time_series_wide.columns = [f'day_{col}_posts' if isinstance(col, int) else col for col in time_series_wide.columns]

print(f"Time series dataset shape: {time_series_wide.shape}")
print(f"Users in time series: {len(time_series_wide)}")
print("\nTime series dataset columns:")
print(time_series_wide.columns.tolist())
print("\nSample of time series data:")
print(time_series_wide.head())


Step 3: Creating wide-format time series dataset...
Time series dataset shape: (39856, 16)
Users in time series: 39856

Time series dataset columns:
['did_id', 'day_0_posts', 'day_1_posts', 'day_2_posts', 'day_3_posts', 'day_4_posts', 'day_5_posts', 'day_6_posts', 'day_7_posts', 'day_8_posts', 'day_9_posts', 'day_10_posts', 'day_11_posts', 'day_12_posts', 'day_13_posts', 'day_14_posts']

Sample of time series data:
   did_id  day_0_posts  day_1_posts  day_2_posts  day_3_posts  day_4_posts  \
0     318          2.0          5.0          4.0          4.0          3.0   
1    1738         19.0          0.0          0.0          0.0          0.0   
2    3316          5.0          0.0          0.0          1.0          0.0   
3    3360          1.0         18.0         11.0          4.0         20.0   
4    6047          0.0          2.0          2.0          0.0          0.0   

   day_5_posts  day_6_posts  day_7_posts  day_8_posts  day_9_posts  \
0          3.0          0.0          3.0 

Create Time series Relative to First Post Date

# Save teh file 

In [8]:
# Save the processed data for future use
output_path = "/home/ale/Documents/uni/mp/data/posting_behaviour/processed/user_activity.parquet"
time_series_wide.to_parquet(output_path, index=False)
print(f"\nData saved to: {output_path}")


Data saved to: /home/ale/Documents/uni/mp/data/posting_behaviour/processed/user_activity.parquet
